In [1]:
import os
import sys
dir_path = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM"
sys.path.insert(0, dir_path)
os.environ['CUDA_VISIBLE_DEVICES'] = "3"
os.environ["HF_HUB_CACHE"]="/qumulo/shared_data/aofei_summer/LLMs"
from llava.eval.cli_v1 import RegLLMChatbot

/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_dir = "/qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_c_45k"
model_args = {
        "model_name_or_path": "Qwen/Qwen3-8B",
        "pretrained_llm_path": model_dir,
        "regtok_config_path": "/qumulo/shared_data/aofei_summer/RegTok/source/tokenizer/regtok_config.yaml",
        "regtok_weight_path": "/qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt",
        "use_regtok": True,
        "mm_vision_vq_type": "RegTok",
        "use_region_tokens": False,
        "vision_tower": "/qumulo/shared_data/aofei_summer/CLIPs/unimed_clip_vit_l14.pt",
        "mm_use_im_start_end": False,
        "mm_use_im_patch_token": True,
        "mm_vision_select_feature": "patch",
        "mm_patch_merge_type": "flat",
        "mm_projector_type": "mlp2x_gelu",
        "pretrain_mm_mlp_adapter": None,
        "mm_vision_select_layer": -1,
        "use_region_tokens": True,
        "use_sep_proj": False,
        "output_segmentation": True,
        "modality_num": 18,
        "codebook_size": 32
    }

In [3]:
bot = RegLLMChatbot(model_dir, model_args=model_args, device="cuda")

loading model from /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_c_45k
use RegSegForCausalLM!
576 codebook_token_ids


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  5.22it/s]
Some weights of the model checkpoint at /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_c_45k were not used when initializing RegSegForCausalLM: ['model.vision_tower.vision_tower.image_encoder.class_embedding', 'model.vision_tower.vision_tower.image_encoder.conv1.weight', 'model.vision_tower.vision_tower.image_encoder.ln_post.bias', 'model.vision_tower.vision_tower.image_encoder.ln_post.weight', 'model.vision_tower.vision_tower.image_encoder.ln_pre.bias', 'model.vision_tower.vision_tower.image_encoder.ln_pre.weight', 'model.vision_tower.vision_tower.image_encoder.positional_embedding', 'model.vision_tower.vision_tower.image_encoder.proj', 'model.vision_tower.vision_tower.image_encoder.transformer.resblocks.0.attn.in_proj_bias', 'model.vision_tower.vision_tower.image_encoder.transformer.resblocks.0.attn.in_proj_weight', 'model.vision_tower.vision_tower.image_encoder.transformer

load vision tower!
Number of stacks: 1
Upsample mode: conv
tokenflow load from: /qumulo/shared_data/aofei_summer/intern_records/RegTok/checkpoints/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt
tokenflow model load success!!
pre loading complete!
Loading segmentation weights from /qumulo/shared_data/aofei_summer/intern_records/LVLM/checkpoints/reg_seg_instruct_mix_c_45k


In [4]:
# bot = RegLLMChatbot(model_dir, model_args=model_args, device="cuda")

In [5]:
bot.inference("Segment the lesions.", images="/qumulo/shared_data/aofei_summer/data/evaluation/imgs/xmlab102/source.jpg")

['assistant\nThe nodule is in the central mid-lung. Segmented as [M5_3], bbox [0.578, 0.613, 0.024, 0.019].']

In [ ]:
bot.generate()

In [6]:
import json
from tqdm import tqdm
model_name = "instruct_regseg_mix_c_45k"
dataset_name = "VQA-RAD"
# question_file = "/qumulo/shared_data/aofei_summer/data/evaluation/test_instruct2.json"
# answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation/inference/answers_{model_name}.jsonl"
# image_folder = "/qumulo/shared_data/aofei_summer/data/evaluation/imgs"

question_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/test_instruct.json"
answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/inference/answers_{model_name}.jsonl"
image_folder = f"/qumulo/shared_data/aofei_summer/data/evaluation_data/{dataset_name}/image"

In [7]:
questions = json.load(open(os.path.expanduser(question_file), "r"))
answers_file = os.path.expanduser(answers_file)
os.makedirs(os.path.dirname(answers_file), exist_ok=True)
ans_file = open(answers_file, "w")
for line in tqdm(questions):

    # idx = line["qid"]
    # question = line["question"] # ['value'].split('\n')[0]
    # gt_ans = line["answer"] # ['value']      
    # image_file = line["img_name"]

    idx = line["id"]
    question = line["conversations"][0]["value"] # ['value'].split('\n')[0]
    gt_ans = line['conversations'][1]['value'] # ['value']
    image_file = line["image"]

    qs = question
    
    image_file = os.path.join(image_folder, image_file)
    ans = bot.inference(qs, image_file)[0]
    ans = ans.replace("assistant\n", "").strip()

    ans_file.write(json.dumps({"question_id": idx,
                                   "prompt": qs,
                                   "text": ans,
                                   "gt_ans": gt_ans,
                                   "metadata": {}}) + "\n")
    ans_file.flush()
ans_file.close()

  0%|          | 0/451 [00:00<?, ?it/s]

100%|██████████| 451/451 [14:02<00:00,  1.87s/it]


In [ ]:
questions = json.load(open(os.path.expanduser(question_file), "r"))
answers_file = os.path.expanduser(answers_file)
os.makedirs(os.path.dirname(answers_file), exist_ok=True)
ans_file = open(answers_file, "w")
for line in tqdm(questions):

    idx = line["qid"]
    question = line["question"] # ['value'].split('\n')[0]
    gt_ans = line["answer"] # ['value']      
    image_file = line["img_name"]

    # idx = line["id"]
    # question = line["conversations"][0]["value"] # ['value'].split('\n')[0]
    # gt_ans = line['conversations'][1]['value'] # ['value']
    # image_file = line["image"]

    qs = question
    
    image_file = os.path.join(image_folder, image_file)
    ans = bot.inference(qs, image_file)[0]
    ans = ans.replace("assistant\n", "").strip()

    ans_file.write(json.dumps({"question_id": idx,
                                   "prompt": qs,
                                   "text": ans,
                                   "gt_ans": gt_ans,
                                   "metadata": {}}) + "\n")
    ans_file.flush()
ans_file.close()

100%|██████████| 1061/1061 [45:51<00:00,  2.59s/it]


In [9]:
(79.92 + 47.16) / 2

63.54